# Lesson 25 - PyTorch and Deep Learning Foundations

**Goals:** Track tensor shapes, differentiate a loss, train/evaluate a tiny network, and reload its weights.

**Before you begin:** Lessons 13–14 cover classes/iterables; Lesson 24 covers supervised learning and holdouts. Previous stage: [24_scikit_learn_classical_ml.ipynb](24_scikit_learn_classical_ml.ipynb).
This lesson defines all its own inputs and runs independently on Python 3.11+.

**Setup:** `requirements.txt` (PyTorch and NumPy); CPU is sufficient. See the README shared requirements instructions. Install from a terminal only when you choose;
these notebooks never install packages. No core example needs a network, API key, database server, or download.
All records are fictional.

Run cells top to bottom with **Shift+Enter**. Predict the output before running each example,
change one input, rerun the relevant cells, and explain what changed. Restart the kernel and
Run All after restoring the original inputs. Practice cells intentionally contain no answers.
Attempt the exercises and project before opening `instructor_solutions/25_pytorch_deep_learning_solutions.ipynb`.


## 1. Tensors, dtypes, and devices

A tensor is an array with optional gradient tracking and device placement. Use floating
features for neural networks, commonly float32; integer class labels often require int64.
The model and data must share a device. We deliberately train on CPU for a small reproducible lab;
CUDA availability is shown without requiring a GPU. Moving a tensor does not move other tensors.
Predict the shape of this three-row, one-feature tensor.


In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(25)
device = torch.device("cpu")
features = torch.tensor([[0.], [0.5], [1.]], dtype=torch.float32, device=device)
print(features.shape, features.dtype, features.device)
print("CUDA available:", torch.cuda.is_available())
print(features * 2, features.mean(dim=0))
assert features.shape == (3,1)


## 2. Shapes are part of the contract

`reshape` changes dimensions without changing element count. `unsqueeze` inserts a size-1
dimension; `squeeze(dim)` removes that dimension only when size 1. Broadcasting resembles NumPy.
For single-output regression keep both predictions and targets `(batch,1)`; comparing `(batch,1)`
with `(batch,)` can accidentally broadcast into `(batch,batch)` and optimize the wrong loss.
`.detach().cpu().numpy()` exports numerical values without gradient tracking.


In [ ]:
flat = torch.tensor([1.,2.,3.])
column = flat.unsqueeze(1)
print(flat.shape, column.shape, (column - flat).shape)
assert (column - flat).shape == (3,3)
assert column.squeeze(1).shape == (3,)
print(column.detach().cpu().numpy().tolist())


## 3. Autograd computes derivatives

A loss is a scalar measure of error. Autograd records operations on tensors requiring
gradients; `backward()` computes how that scalar changes with each parameter.
For `(w*x-y)^2` at w=1, x=2, y=6, the derivative is -16. Gradients accumulate, so clear them
between optimizer steps. The gradient is a direction signal, not a new parameter value.


In [ ]:
w = torch.tensor(1., requires_grad=True)
loss = (w * 2 - 6) ** 2
loss.backward()
print(loss.item(), w.grad.item())
assert loss.item() == 16 and w.grad.item() == -16
w.grad.zero_()


## 4. Dataset and DataLoader separate rows from batches

A Dataset supplies length and indexed samples. A DataLoader forms mini-batches, optionally
shuffling training rows each epoch. We use num_workers=0 for notebook/Windows simplicity.
Generate noiseless normalized features and a nonlinear target. The fixed split interleaves held-out
points across the interval; no test values are used to fit parameters. Real temporal/grouped data
needs an appropriate split as in Lesson 24.


In [ ]:
class EnergyDataset(Dataset):
    def __init__(self, x, y):
        if x.ndim != 2 or y.shape != (len(x),1):
            raise ValueError("Expected features (n,f) and target (n,1)")
        self.x, self.y = x, y
    def __len__(self):
        return len(self.x)
    def __getitem__(self, index):
        return self.x[index], self.y[index]

x = torch.linspace(0,1,80).reshape(-1,1)
y = 0.2 + 0.5*x + 0.3*x**2  # Normalized synthetic energy, not physical calibration.
test_mask = torch.arange(80) % 4 == 0
train_data = EnergyDataset(x[~test_mask], y[~test_mask])
test_data = EnergyDataset(x[test_mask], y[test_mask])
loader = DataLoader(train_data, batch_size=20, shuffle=True, num_workers=0,
                    generator=torch.Generator().manual_seed(25))
batch_x, batch_y = next(iter(loader))
print(len(train_data), len(test_data), batch_x.shape, batch_y.shape)
assert len(train_data) == 60 and len(test_data) == 20


## 5. A tiny network and its loss

`nn.Linear(1,8)` maps one input feature to eight hidden values. ReLU keeps positive values
and replaces negatives with zero, adding nonlinearity. A final Linear maps eight hidden values
to one prediction. Sequential composes layers; it is a concrete use of earlier composition lessons.
MSELoss averages squared errors. Adam adjusts parameters using gradients and a learning rate;
too large a rate can make training unstable. This network is deliberately small.


In [ ]:
def make_model():
    return nn.Sequential(nn.Linear(1,8), nn.ReLU(), nn.Linear(8,1))

torch.manual_seed(25)
model = make_model().to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
assert model(batch_x.to(device)).shape == batch_y.shape
with torch.no_grad():
    initial_loss = loss_fn(model(train_data.x), train_data.y).item()
print("Train loss before fitting:", round(initial_loss,4))


## 6. Train one small batch at a time

Each epoch visits the training set once. `train()` enables training behavior for layers
such as dropout (not used here). For each batch: clear old gradients, predict, compute a scalar
loss, backpropagate, and update parameters. `item()` turns a one-value tensor into a Python number
for reporting. We report a final whole-training-set loss rather than printing every batch.
Change the epoch count after making a fresh model; explain the effect.


In [ ]:
for epoch in range(100):
    model.train()
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        predictions = model(inputs)
        assert predictions.shape == targets.shape
        batch_loss = loss_fn(predictions, targets)
        batch_loss.backward()
        optimizer.step()

with torch.no_grad():
    final_loss = loss_fn(model(train_data.x), train_data.y).item()
print("Training MSE:", round(initial_loss,5), "->", round(final_loss,5))
assert torch.isfinite(torch.tensor(final_loss))
assert final_loss < initial_loss  # Relative learning check, not an exact cross-platform weight check.


## 7. Evaluate without learning from test data

`eval()` changes layer behavior; it does not turn off gradient recording. `no_grad()` turns
off that recording for this block. Use both for inference. Do not call backward or optimizer.step
on held-out rows. MAE below is in normalized target units. Compare with a constant training-mean
baseline before claiming the network is useful. Small differences across devices/versions are normal.


In [ ]:
model.eval()
with torch.no_grad():
    test_prediction = model(test_data.x.to(device))
    test_mae = (test_prediction - test_data.y.to(device)).abs().mean().item()
    baseline_mae = (train_data.y.mean() - test_data.y).abs().mean().item()
print({"test_MAE": round(test_mae,4), "baseline_MAE": round(baseline_mae,4)})
assert test_prediction.shape == (20,1)
assert torch.isfinite(test_prediction).all()


## 8. Save weights and reconstruct the architecture

A state_dict stores learned parameters; reconstruct the same architecture before loading.
Save inside TemporaryDirectory so the lesson leaves no model file behind. Use `weights_only=True`
explicitly and load only trusted artifacts. In a real project, version the architecture, feature
order, normalization, and training metadata alongside the weights. A weight file alone does not
describe the full serving contract.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    path = Path(folder) / "energy_weights.pt"
    torch.save(model.state_dict(), path)
    restored = make_model().to(device)
    restored.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    restored.eval()
    with torch.no_grad():
        restored_prediction = restored(test_data.x)
    torch.testing.assert_close(restored_prediction, test_prediction)
print("Weights round trip preserves predictions.")


## 9. Common mistakes and debugging

Print tensor shape, dtype, device, and finite status at each boundary. A dtype mismatch usually
means integer/double inputs reached float32 weights. A device mismatch means data and model differ.
A loss that barely moves may reflect missing optimizer.step, detached predictions, an unsuitable
learning rate, or a shape broadcast. Forgetting zero_grad accumulates old gradients. Repeatedly
checking test error to choose epochs turns the test set into validation data; use a separate
validation split for tuning. A neural network is not automatically better than Lesson 24's simple models.


## Practice lab

Use the supplied inputs, add your own comments, and compare with the check values. Use `practice_`
names where helpful. Increase the difficulty one exercise at a time. Checks describe results;
they are not complete implementations. Test a boundary or malformed input when applicable.


### Exercise 1. Make a feature column

Convert `[1,2,3,4]` to float32 shape `(4,1)` on CPU.

**Check:** ndim 2, dtype float32, device cpu.


In [ ]:
practice_values = [1,2,3,4]
# TODO: tensor, dtype, reshape, inspect.


### Exercise 2. Differentiate a scalar

For w=2 and loss `(3*w-9)**2`, compute the loss and gradient.

**Check:** loss 9, gradient -18. Explain the chain rule in words.


In [ ]:
practice_w = torch.tensor(2., requires_grad=True)
# TODO: build loss then backward.


### Exercise 3. Inspect batching

Use EnergyDataset for 10 rows and a DataLoader with batch size 4, shuffle False.

**Check:** batch lengths `[4,4,2]`, each feature/target has one column.


In [ ]:
practice_x = torch.arange(10, dtype=torch.float32).reshape(-1,1)
# TODO: dataset, loader, inspect all batch shapes.


### Exercise 4. One optimizer step

Use scalar w=0, SGD learning rate 0.1, loss `(2*w-4)**2`. Perform one update.

**Check:** w approximately 1.6 and loss decreases from 16 to 0.64.


In [ ]:
practice_w = nn.Parameter(torch.tensor(0.))
# TODO: SGD, zero gradients, backward, step, recalculate loss.


### Exercise 5. Round-trip an inference model

Create a fresh make_model, save/reload its state_dict in a temporary folder, and compare predictions on `[[0],[1]]` in eval/no_grad.

**Check:** all predictions close; shape `(2,1)`; no persistent file.


In [ ]:
practice_inputs = torch.tensor([[0.],[1.]])
# TODO: fresh model, temporary artifact, same architecture, inference comparison.


## Chapter project

Wrap inference in `predict_energy(model, hours)` for a nonempty list of finite normalized hours
between 0 and 1. Convert to float32 `(n,1)`, reject booleans and invalid values, use eval/no_grad,
and return a list of floats without changing weights. Use a known nn.Linear with weight 2 and bias 1
to test the wrapper independently of stochastic training.

**Check:** `[0,0.5,1]` gives `[1,2,3]`; empty, negative, NaN, and boolean inputs fail.
Then apply it to your trained network and describe the normalized output units.


In [ ]:
def predict_energy(model, hours):
    # TODO: validate boundary, move to model device, infer, export.
    pass


## Reflection and knowledge check

1. Why must prediction and target shapes match exactly?
2. What do zero_grad, backward, and step each do?
3. How do eval and no_grad differ?
4. What information must accompany a saved state_dict?

Choose one changed input, predict the new result, and explain your observation to a colleague.


## Official references

- [PyTorch basics](https://docs.pytorch.org/tutorials/beginner/basics/)
- [Save and load weights](https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html)
